# 07 · Escenarios y sensibilidad

Cuantifica la exposición del EBITDA a ocupación, precio, combustible y eficiencia, sin presentar escenarios como previsiones.

> **Fuente:** datos sintéticos de Levante Ferries. Proyecto demostrativo; no contiene información real de ninguna naviera.

## Contexto y método

El notebook forma parte de una cadena reproducible. Las fórmulas y supuestos se muestran junto a los resultados para que cada conclusión pueda revisarse.

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
for folder in [PROCESSED, TABLES, FIGURES]: folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#1f6feb','#2dd4bf','#f59e0b','#ef4444','#94a3b8'])
plt.rcParams.update({'figure.figsize': (11, 5.5), 'axes.titlesize': 14, 'axes.labelsize': 10})
pd.options.display.float_format = '{:,.2f}'.format


## Base de 2026

In [2]:
a=pd.read_csv(RAW/'fact_finance_actual.csv',parse_dates=['month'])
base=a[a.year.eq(2026)].copy()
base_ebitda=base.ebitda.sum()
scenarios=[('Base',0,0,0,0),('Fuel +15%',0,0,.15,0),('Demanda -8%',-.08,0,0,0),('Precio +3%',0,.03,0,0),('Eficiencia combinada',.02,.02,-.06,-.03)]
rows=[]
for name,demand,price,fuel,other_cost in scenarios:
    rev=base.revenue.sum()*(1+demand)*(1+price)
    fuel_c=base.fuel_cost.sum()*(1+demand)*(1+fuel)
    nonfuel_var=(base.variable_cost.sum()-base.fuel_cost.sum())*(1+demand)*(1+other_cost)
    ebitda=rev-fuel_c-nonfuel_var-base.fixed_allocated.sum()
    rows.append({'scenario':name,'revenue':rev,'ebitda':ebitda,'ebitda_margin':ebitda/rev,'delta_vs_base':ebitda-base_ebitda})
scenario=pd.DataFrame(rows)
scenario.to_csv(TABLES/'07_scenarios_2026.csv',index=False)
display(scenario)

               scenario        revenue  ...  ebitda_margin  delta_vs_base
0                  Base 200,111,488.76  ...           0.10           0.00
1             Fuel +15% 200,111,488.76  ...           0.04 -12,019,184.23
2           Demanda -8% 184,102,569.66  ...           0.09  -3,915,329.70
3            Precio +3% 206,114,833.42  ...           0.13   6,003,344.66
4  Eficiencia combinada 208,195,992.91  ...           0.15  12,138,818.32

[5 rows x 5 columns]


## Matriz ocupación–combustible

In [3]:
grid=[]
for occ in [-.10,-.05,0,.05,.10]:
 for fuel in [-.15,-.075,0,.075,.15]:
  rev=base.revenue.sum()*(1+occ)
  costs=(base.variable_cost.sum()-base.fuel_cost.sum())*(1+occ)+base.fuel_cost.sum()*(1+occ)*(1+fuel)+base.fixed_allocated.sum()
  grid.append({'demand_change':occ,'fuel_change':fuel,'ebitda_margin':(rev-costs)/rev})
grid=pd.DataFrame(grid); grid.to_csv(TABLES/'07_sensitivity_grid.csv',index=False)
pivot=grid.pivot(index='demand_change',columns='fuel_change',values='ebitda_margin')*100
plt.figure(figsize=(9,6)); sns.heatmap(pivot,annot=True,fmt='.1f',cmap='RdYlGn',center=12,cbar_kws={'label':'Margen EBITDA %'}); plt.xlabel('Variación combustible'); plt.ylabel('Variación demanda'); plt.title('Sensibilidad del margen EBITDA 2026')
plt.tight_layout(); plt.savefig(FIGURES/'07_sensitivity_heatmap.png',dpi=180,bbox_inches='tight'); plt.show()

## Conclusiones

Las conclusiones concretas se generan a partir de las salidas ejecutadas. Deben interpretarse como evidencia de una simulación y como demostración del método analítico.